# CPC353 Assignment 1 — Stop-Word Analysis

This notebook analyzes the project stop-word list in the Shin Ramyun review corpus. It compares collected reviews and AI-generated examples separately, and keeps the original wording and unfiltered tokens available.

Run preprocessing.ipynb first. This notebook saves two CSV tables in data/analysis/.


## 1. Imports and locate the project folder

The notebook searches the current folder and its parents for the preprocessing outputs, so it works when launched from the repository root or a subfolder.


In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

PROJECT_DIR = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "data" / "processed").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find data/processed. Run this notebook from the project.")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "data" / "analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import display
except ImportError:
    display = print

print(f"Project folder: {PROJECT_DIR}")
print(f"Analysis outputs: {OUTPUT_DIR}")

## 2. Load the retained reviews and project stop words

Only included reviews are counted. The origin and review language labels are retained as grouping fields. The language column describes each review's label; it does not automatically identify the language of every individual token.


In [ ]:
def load_jsonl(path):
    with path.open(encoding="utf-8") as file:
        rows = [json.loads(line) for line in file if line.strip()]
    return [row for row in rows if row.get("included", True)]

reviews_by_origin = {
    "real": load_jsonl(PROCESSED_DIR / "real_preprocessed.jsonl"),
    "ai": load_jsonl(PROCESSED_DIR / "ai_preprocessed.jsonl"),
}

stopwords_df = pd.read_csv(PROCESSED_DIR / "stopwords_used.csv")
stopwords_df["word"] = stopwords_df["word"].astype(str).str.casefold()
stopwords_df["language"] = stopwords_df["language"].astype(str)

stopword_language = dict(zip(stopwords_df["word"], stopwords_df["language"]))
stopword_set = set(stopword_language)

print("Included review counts:")
display(pd.DataFrame([
    {"origin": origin, "reviews": len(rows)}
    for origin, rows in reviews_by_origin.items()
]))
print(f"Project stop words loaded: {len(stopword_set)}")

## 3. Compare stop-word frequencies

For every project stop word, the table records its count before filtering, how many occurrences were removed, and how many remain afterward. Counts are also normalized per 1,000 unfiltered tokens so the two origins can be compared despite different corpus sizes.

The preprocessing list is project-authored and conservative. Negators such as not, tak, and bukan are intentionally protected and will not appear as removed stop words.


In [ ]:
def counts_for(rows, field):
    counts = Counter()
    for row in rows:
        counts.update(str(token).casefold() for token in row.get(field, []))
    return counts

frequency_rows = []
for origin, rows in reviews_by_origin.items():
    before = counts_for(rows, "tokens_before")
    removed = counts_for(rows, "tokens_removed")
    after = counts_for(rows, "tokens_after")
    total_before = sum(before.values())

    for word in sorted(stopword_set):
        frequency_rows.append({
            "origin": origin,
            "stopword_language": stopword_language[word],
            "word": word,
            "before_count": before[word],
            "removed_count": removed[word],
            "after_count": after[word],
            "before_per_1000_tokens": round(before[word] / total_before * 1000, 3)
                if total_before else 0,
            "reviews_containing_word": sum(
                word in {str(token).casefold() for token in row.get("tokens_before", [])}
                for row in rows
            ),
            "total_unfiltered_tokens": total_before,
        })

stopword_frequency_df = pd.DataFrame(frequency_rows)
frequency_path = OUTPUT_DIR / "stopword_frequency_by_origin.csv"
stopword_frequency_df.to_csv(frequency_path, index=False, encoding="utf-8-sig")

print("Most frequent project stop words before filtering, by origin:")
display(
    stopword_frequency_df.sort_values(
        ["origin", "before_count", "word"], ascending=[True, False, True]
    ).groupby("origin", group_keys=False).head(20)
)

print("Check: stop-word occurrences remaining after filtering:")
display(
    stopword_frequency_df.groupby("origin", as_index=False)["after_count"].sum()
)

## 4. Review-level stop-word use

This table shows how many stop-word occurrences each review had and what share of its unfiltered tokens they represented. It helps distinguish a frequent word from a review that simply contains many tokens.


In [ ]:
review_rows = []
for origin, rows in reviews_by_origin.items():
    for row in rows:
        before_tokens = [str(token).casefold() for token in row.get("tokens_before", [])]
        removed_tokens = [str(token).casefold() for token in row.get("tokens_removed", [])]
        total_tokens = len(before_tokens)
        removed_english = sum(
            stopword_language.get(token) == "English" for token in removed_tokens
        )
        removed_malay = sum(
            stopword_language.get(token) == "Malay" for token in removed_tokens
        )
        review_rows.append({
            "origin": origin,
            "review_id": row.get("review_id", ""),
            "review_language": row.get("language", ""),
            "unfiltered_token_count": total_tokens,
            "stopword_occurrences_before": sum(token in stopword_set for token in before_tokens),
            "stopword_occurrences_removed": len(removed_tokens),
            "english_stopword_occurrences_removed": removed_english,
            "malay_stopword_occurrences_removed": removed_malay,
            "removed_share_percent": round(len(removed_tokens) / total_tokens * 100, 2)
                if total_tokens else 0,
        })

review_stopword_df = pd.DataFrame(review_rows)
review_path = OUTPUT_DIR / "stopword_review_summary.csv"
review_stopword_df.to_csv(review_path, index=False, encoding="utf-8-sig")

print("Review-level stop-word summary:")
display(review_stopword_df.head(20))
print("Mean percentage of tokens removed by origin:")
display(
    review_stopword_df.groupby("origin", as_index=False)["removed_share_percent"].mean()
)

## 5. Interpretation notes and saved outputs

The notebook creates:
- data/analysis/stopword_frequency_by_origin.csv
- data/analysis/stopword_review_summary.csv

The original reviews and tokens_before remain intact. Use before_count to describe stop-word use, removed_count to confirm what the filter removed, and the normalized rate when comparing real and AI corpus sizes. The AI rows are synthetic and must be discussed separately from collected reviews. These tables are CSV files that Excel can open; they are not Excel workbooks.
